# EVE 310 - Lab 11: Comparing regression models — activity

**Module 2 | 11/12/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab11-dairy-farm-challenge/notebooks/lab11-activity.ipynb)

## Learning objectives

Fit and score a linear model and a forest; judge whether a column built from units lets the linear model beat the forest.

<details><summary>More: the three objectives</summary>

By the end of this lab you will be able to:

1. Fit a linear model and a random forest on the same training rows, and read the linear model's betas in units
2. Score both on the testing rows with the MAE, compare them with a model that knows nothing, and check the forest for overfitting
3. Build a new column from the units of two others, and judge whether it lets the linear model beat the forest

</details>

## Before you start

1. Click **Copy to Drive** (top of this window) and work in the copy; uncopied work is lost.
2. Run the setup cell first.
3. Work in order, keeping the given variable names. Replace each `--`: one line in Questions 1–2, a whole step in Question 3. Try first; stuck? **Show the finished cell**.
4. Keep `random_state=25` (split) and `random_state=0` (forest), so everyone gets the numbers the autograder and quiz use.
5. Upload this notebook to Gradescope (steps in the Wrap-up): feedback only, **not graded**, unlimited uploads. The Lab 11 Canvas quiz counts and asks about your results here: finish the activity first, at home if needed.

<details><summary>More: setup and grading</summary>

Colab throws away anything not copied to Drive when the runtime ends. The setup cell downloads the two data files into `DATA_DIR` and creates `FIGURES_DIR`. The three short questions follow the tutorial's plan: fit, check, then prepare, fit and check a model with one new column. In Question 3 a `--` stands for as many lines as the step takes. **Show the finished cell** under each question has the answer. The autograder gives feedback on each question.

</details>

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab11-dairy-farm-challenge"
DATA_FILES = ["power_demand.csv", "Dairy-Farm-Dataset.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**The data (run as is)**

`Dairy-Farm-Dataset.csv`: 10,000 dairy farms (synthetic). A dairy cooperative asks which practices lower `Carbon_footprint`, and by how much.

<details><summary>More: emissions and units</summary>

Producing milk emits greenhouse gases: methane from the cows and their manure, nitrous oxide from fertilized fields, carbon dioxide from feed production and energy. `Carbon_footprint` is the emissions per kg of milk, and each row is one farm. Milk here means fat-and-protein-corrected milk (FPCM). CO₂e (CO₂-equivalent) adds up all greenhouse gases, each weighted by how strongly it warms the climate.

</details>

| Column | Meaning | Units |
| --- | --- | --- |
| `Feed_inefficiency` | feed eaten | kg feed per kg milk |
| `Protein_efficiency` | share of eaten nitrogen that ends up in milk, growth, beef | % |
| `Animal_mortality` | deaths that year | % of the herd |
| `Fertilizer_use` | nitrogen spread as manure and fertilizer | kg N per hectare |
| `Land_use` | land to grow all feed | m² per kg milk |
| `Soy_use` | soy in feed | g per kg milk |
| `Permanent_grassland` | permanent grassland | hectares per cow |
| `Grazing` | grazing ≥ 6 h/day, 120 days/year | % of cows |
| `Manure_biogas` | 1 if 75%+ of manure goes to biogas or acidified slurry | 0 or 1 |
| `Renewable_electricity_use` | 1 if all electricity is wind, solar, biogas or hydro | 0 or 1 |
| `Carbon_footprint` | to predict | kg CO₂e per kg milk |

Fixed split: 70% for fitting, 30% for testing (`random_state=25`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

farm_df = pd.read_csv(DATA_DIR / 'Dairy-Farm-Dataset.csv')
print(farm_df.shape)
# The mean, smallest and largest value of every column, one row per column
print(farm_df.describe().loc[['mean', 'min', 'max']].T.round(3))

y = farm_df['Carbon_footprint']
x = farm_df.drop(columns=['Carbon_footprint'])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=25)
print(len(x_train), 'training farms,', len(x_test), 'testing farms')

## Your turn

**Question 1 — Fit: a linear model and a forest on all ten columns**

Fit each model in one line: chain `.fit(...)`, as in Lab 10's `KMeans(...).fit(X)`.

- `lm_all`: `LinearRegression` fitted on `x_train`, `y_train`
- `betas`: its betas, a pandas Series labeled by column name (`x_train.columns`), as in Lab 6
- `cut_both`: predicted footprint change when a farm switches both `Manure_biogas` and `Renewable_electricity_use` from 0 to 1 (both betas from `betas`, by name)
- `rf`: `RandomForestRegressor(random_state=0)` fitted on the same rows (10 to 20 s)

Checks: `lm_all` and `betas` against the normal equation on the 7,000 training farms; `cut_both` against one farm's predictions.

<details><summary>More: units and checks</summary>

`cut_both` is in kg CO₂e per kg of milk; its check compares it with what `lm_all` predicts for one farm switching both practices on. The normal equation uses a design matrix of 11 columns: the ones, then the ten columns.

</details>

In [ ]:
# A linear model on all ten columns, created and fitted in one line
lm_all = --
# Its betas, one per column, labeled by the column names
betas = --
# The predicted change for a farm that switches to biogas and to renewable electricity
cut_both = --
# A random forest on the same rows (keep random_state=0)
rf = --

print(betas.round(4))
print('intercept:', round(lm_all.intercept_, 4))
print('both practices:', round(cut_both, 4), 'kg CO2e per kg of milk')

# The normal equation for the 7,000 training farms: a column of ones, then the ten columns
X1 = np.c_[np.ones(len(x_train)), x_train]
beta_check = np.dot(np.linalg.inv(np.dot(X1.T, X1)), np.dot(X1.T, y_train))
print('The normal equation gives the intercept and the betas, labeled by column?',
      np.isclose(beta_check[0], lm_all.intercept_) and np.allclose(beta_check[1:], betas)
      and list(getattr(betas, 'index', [])) == list(x_train.columns))
# One farm with both practices, and the same farm with neither
farm_both = x_train.iloc[[0]].assign(Manure_biogas=1, Renewable_electricity_use=1)
farm_neither = x_train.iloc[[0]].assign(Manure_biogas=0, Renewable_electricity_use=0)
print("cut_both is the change in that farm's prediction, and the forest saw the ten columns with random_state=0?",
      np.ndim(cut_both) == 0 and np.isclose(cut_both, lm_all.predict(farm_both)[0] - lm_all.predict(farm_neither)[0])
      and getattr(rf, 'n_features_in_', 0) == 10 and rf.random_state == 0)

<details><summary>Show the finished cell</summary>

```python
# A linear model on all ten columns, created and fitted in one line
lm_all = LinearRegression().fit(x_train, y_train)
# Its betas, one per column, labeled by the column names
betas = pd.Series(lm_all.coef_, index=x_train.columns)
# The predicted change for a farm that switches to biogas and to renewable electricity
cut_both = betas['Manure_biogas'] + betas['Renewable_electricity_use']
# A random forest on the same rows (keep random_state=0)
rf = RandomForestRegressor(random_state=0).fit(x_train, y_train)

print(betas.round(4))
print('intercept:', round(lm_all.intercept_, 4))
print('both practices:', round(cut_both, 4), 'kg CO2e per kg of milk')

# The normal equation for the 7,000 training farms: a column of ones, then the ten columns
X1 = np.c_[np.ones(len(x_train)), x_train]
beta_check = np.dot(np.linalg.inv(np.dot(X1.T, X1)), np.dot(X1.T, y_train))
print('The normal equation gives the intercept and the betas, labeled by column?',
      np.isclose(beta_check[0], lm_all.intercept_) and np.allclose(beta_check[1:], betas)
      and list(getattr(betas, 'index', [])) == list(x_train.columns))
# One farm with both practices, and the same farm with neither
farm_both = x_train.iloc[[0]].assign(Manure_biogas=1, Renewable_electricity_use=1)
farm_neither = x_train.iloc[[0]].assign(Manure_biogas=0, Renewable_electricity_use=0)
print("cut_both is the change in that farm's prediction, and the forest saw the ten columns with random_state=0?",
      np.ndim(cut_both) == 0 and np.isclose(cut_both, lm_all.predict(farm_both)[0] - lm_all.predict(farm_neither)[0])
      and getattr(rf, 'n_features_in_', 0) == 10 and rf.random_state == 0)
```

</details>

Which of the two practices cuts the footprint more, according to `betas`? Why can the two betas be added for a farm adopting both?

*Your answer:*

**Question 2 — Check: score both models on the testing farms**

Score the given predictions by MAE, truth first: `mean_absolute_error(y_test, ...)` or `np.mean(np.abs(...))`.

- `mae_lm`, `mae_rf`: each model's testing MAE
- `mae_train_rf`: the forest's training MAE (the linear model's is printed too)
- `mae_flat`: testing MAE of a know-nothing model predicting the training farms' mean footprint for every testing farm

The check recomputes each MAE by hand.

<details><summary>More: predictions and units</summary>

The predictions made for you are on the testing farms (`y_pred_...`) and on the training farms (`y_fit_...`). `mae_lm` and `mae_rf` are in kg CO₂e per kg of milk.

</details>

In [ ]:
# Predictions on the testing farms and on the training farms, made for you
y_pred_lm = lm_all.predict(x_test)
y_pred_rf = rf.predict(x_test)
y_fit_lm = lm_all.predict(x_train)
y_fit_rf = rf.predict(x_train)

# The testing MAE of each model, in kg CO2e per kg of milk
mae_lm = --
mae_rf = --
# The forest's MAE on the farms it was fitted on
mae_train_rf = --
# A model that knows nothing: every testing farm predicted with the mean of the training farms
mae_flat = --

print('knows nothing: MAE testing', round(mae_flat, 4))
print('linear:        MAE testing', round(mae_lm, 4), '  training', round(mean_absolute_error(y_train, y_fit_lm), 4))
print('forest:        MAE testing', round(mae_rf, 4), '  training', round(mae_train_rf, 4))
print('Each MAE is one number, the mean absolute error on the right farms?',
      all(np.ndim(v) == 0 for v in (mae_lm, mae_rf, mae_train_rf, mae_flat))
      and np.isclose(mae_lm, np.mean(np.abs(y_test - y_pred_lm))) and np.isclose(mae_rf, np.mean(np.abs(y_test - y_pred_rf)))
      and np.isclose(mae_train_rf, np.mean(np.abs(y_train - y_fit_rf)))
      and np.isclose(mae_flat, np.mean(np.abs(y_test - np.mean(y_train)))))

# Predicted against true footprint for the testing farms; on the dashed line, prediction = truth
fig_pred, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(y_test, y_pred_lm, s=4)
ax1.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax1.set_title('Linear model, ten columns')
ax2.scatter(y_test, y_pred_rf, s=4, color='tab:orange')
ax2.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax2.set_title('Random forest, ten columns')
for ax in (ax1, ax2):
    ax.set_xlabel('true footprint (kg CO2e per kg milk)')
    ax.set_ylabel('predicted footprint')
fig_pred.savefig(FIGURES_DIR / 'lab11_predicted_vs_true.png', dpi=150)

<details><summary>Show the finished cell</summary>

```python
# Predictions on the testing farms and on the training farms, made for you
y_pred_lm = lm_all.predict(x_test)
y_pred_rf = rf.predict(x_test)
y_fit_lm = lm_all.predict(x_train)
y_fit_rf = rf.predict(x_train)

# The testing MAE of each model, in kg CO2e per kg of milk
mae_lm = mean_absolute_error(y_test, y_pred_lm)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
# The forest's MAE on the farms it was fitted on
mae_train_rf = mean_absolute_error(y_train, y_fit_rf)
# A model that knows nothing: every testing farm predicted with the mean of the training farms
mae_flat = np.mean(np.abs(y_test - np.mean(y_train)))

print('knows nothing: MAE testing', round(mae_flat, 4))
print('linear:        MAE testing', round(mae_lm, 4), '  training', round(mean_absolute_error(y_train, y_fit_lm), 4))
print('forest:        MAE testing', round(mae_rf, 4), '  training', round(mae_train_rf, 4))
print('Each MAE is one number, the mean absolute error on the right farms?',
      all(np.ndim(v) == 0 for v in (mae_lm, mae_rf, mae_train_rf, mae_flat))
      and np.isclose(mae_lm, np.mean(np.abs(y_test - y_pred_lm))) and np.isclose(mae_rf, np.mean(np.abs(y_test - y_pred_rf)))
      and np.isclose(mae_train_rf, np.mean(np.abs(y_train - y_fit_rf)))
      and np.isclose(mae_flat, np.mean(np.abs(y_test - np.mean(y_train)))))

# Predicted against true footprint for the testing farms; on the dashed line, prediction = truth
fig_pred, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(y_test, y_pred_lm, s=4)
ax1.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax1.set_title('Linear model, ten columns')
ax2.scatter(y_test, y_pred_rf, s=4, color='tab:orange')
ax2.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax2.set_title('Random forest, ten columns')
for ax in (ax1, ax2):
    ax.set_xlabel('true footprint (kg CO2e per kg milk)')
    ax.set_ylabel('predicted footprint')
fig_pred.savefig(FIGURES_DIR / 'lab11_predicted_vs_true.png', dpi=150)
```

</details>

What share of the know-nothing error does each model remove? Which model overfits, and how do its two MAEs show it? Does that change which predicts the testing farms better?

*Your answer:*

**Question 3 — Prepare, fit, check: one new column from the units**

Nitrous oxide from fertilized fields depends on nitrogen spread **per kg of milk**. Multiply `Fertilizer_use` (kg N per hectare) by `Land_use` (m² per kg milk), converting m² to hectares (1 hectare = 10,000 m²): the units cancel to kg N per kg milk.

- `x_train_n`, `x_test_n`: the given copies plus `N_per_kg_milk` (eleven columns; `x_train`, `x_test` keep ten)
- `lm_n`: `LinearRegression` fitted on `x_train_n`
- `beta_n`: its beta for `N_per_kg_milk`
- `mae_n`: its testing MAE

Write each step under its comment. The units check catches a power-of-ten conversion error. First check False because the column went into `x_train` or `x_test`? Rerun the data cell (it rebuilds the split), then this cell.

In [ ]:
# Copies of the split, made for you: x_train and x_test keep their ten columns
x_train_n = x_train.copy()
x_test_n = x_test.copy()

# Each placeholder line below is a whole step: replace it with as many lines as the step needs

# Step 1, prepare: add N_per_kg_milk, kg of nitrogen per kg of milk, to x_train_n and to x_test_n
--

# Step 2, fit: lm_n, a linear model fitted on x_train_n, and beta_n, its beta for N_per_kg_milk
--

# Step 3, check: mae_n, the testing MAE of lm_n
--

# Step 4: the checks and the three models side by side, written for you
has_n = 'N_per_kg_milk' in x_train_n.columns and 'N_per_kg_milk' in x_test_n.columns
print('N_per_kg_milk in both copies, eleven columns each, ten still in x_train and x_test?',
      has_n and x_train_n.shape[1] == 11 and x_test_n.shape[1] == 11 and x_train.shape[1] == 10 and x_test.shape[1] == 10)
print('N_per_kg_milk in kg of nitrogen per kg of milk (every value between 0.001 and 0.5)?',
      has_n and bool(x_train_n['N_per_kg_milk'].between(0.001, 0.5).all() and x_test_n['N_per_kg_milk'].between(0.001, 0.5).all()))
# A least-squares fit leaves X-transpose times its residuals at zero (tutorial section 2), on the rows it was fitted on
print('lm_n fitted on the training farms, beta_n its N_per_kg_milk beta, and mae_n its testing MAE?',
      has_n and np.ndim(beta_n) == 0 and np.ndim(mae_n) == 0
      and np.allclose(np.dot(x_train_n.T, y_train - lm_n.predict(x_train_n)), 0, atol=1e-3)
      and np.isclose(beta_n, lm_n.coef_[list(x_train_n.columns).index('N_per_kg_milk')])
      and np.isclose(mae_n, np.mean(np.abs(y_test - lm_n.predict(x_test_n)))))
scores_df = pd.DataFrame({'model': ['linear, ten columns', 'random forest, ten columns', 'linear, eleven columns'],
                          'MAE testing': [mae_lm, mae_rf, mae_n],
                          'R2 testing': [r2_score(y_test, y_pred_lm), r2_score(y_test, y_pred_rf),
                                         r2_score(y_test, lm_n.predict(x_test_n))]})
print(scores_df.round(4))
print('beta for N_per_kg_milk:', round(beta_n, 3))

<details><summary>Show the finished cell</summary>

```python
# Copies of the split, made for you: x_train and x_test keep their ten columns
x_train_n = x_train.copy()
x_test_n = x_test.copy()

# Each placeholder line below is a whole step: replace it with as many lines as the step needs

# Step 1, prepare: add N_per_kg_milk, kg of nitrogen per kg of milk, to x_train_n and to x_test_n
x_train_n['N_per_kg_milk'] = x_train_n['Fertilizer_use'] * x_train_n['Land_use'] / 10000
x_test_n['N_per_kg_milk'] = x_test_n['Fertilizer_use'] * x_test_n['Land_use'] / 10000

# Step 2, fit: lm_n, a linear model fitted on x_train_n, and beta_n, its beta for N_per_kg_milk
lm_n = LinearRegression().fit(x_train_n, y_train)
beta_n = pd.Series(lm_n.coef_, index=x_train_n.columns)['N_per_kg_milk']

# Step 3, check: mae_n, the testing MAE of lm_n
mae_n = mean_absolute_error(y_test, lm_n.predict(x_test_n))

# Step 4: the checks and the three models side by side, written for you
has_n = 'N_per_kg_milk' in x_train_n.columns and 'N_per_kg_milk' in x_test_n.columns
print('N_per_kg_milk in both copies, eleven columns each, ten still in x_train and x_test?',
      has_n and x_train_n.shape[1] == 11 and x_test_n.shape[1] == 11 and x_train.shape[1] == 10 and x_test.shape[1] == 10)
print('N_per_kg_milk in kg of nitrogen per kg of milk (every value between 0.001 and 0.5)?',
      has_n and bool(x_train_n['N_per_kg_milk'].between(0.001, 0.5).all() and x_test_n['N_per_kg_milk'].between(0.001, 0.5).all()))
# A least-squares fit leaves X-transpose times its residuals at zero (tutorial section 2), on the rows it was fitted on
print('lm_n fitted on the training farms, beta_n its N_per_kg_milk beta, and mae_n its testing MAE?',
      has_n and np.ndim(beta_n) == 0 and np.ndim(mae_n) == 0
      and np.allclose(np.dot(x_train_n.T, y_train - lm_n.predict(x_train_n)), 0, atol=1e-3)
      and np.isclose(beta_n, lm_n.coef_[list(x_train_n.columns).index('N_per_kg_milk')])
      and np.isclose(mae_n, np.mean(np.abs(y_test - lm_n.predict(x_test_n)))))
scores_df = pd.DataFrame({'model': ['linear, ten columns', 'random forest, ten columns', 'linear, eleven columns'],
                          'MAE testing': [mae_lm, mae_rf, mae_n],
                          'R2 testing': [r2_score(y_test, y_pred_lm), r2_score(y_test, y_pred_rf),
                                         r2_score(y_test, lm_n.predict(x_test_n))]})
print(scores_df.round(4))
print('beta for N_per_kg_milk:', round(beta_n, 3))
```

</details>

Say what `beta_n` means in words, with units: how much does the predicted footprint change when `N_per_kg_milk` rises 0.01 kg (10 g), every other column unchanged?

*Your answer:*

**Quiz scratch space**

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

**Runtime > Restart session and run all**, confirm every cell runs, then **File > Download > Download .ipynb** and upload to Gradescope. Keep this notebook open for the Canvas quiz.